# PC1 size across OxCGRT countries

This notebook presents the same PCA analysis as in `06-oxcgrt-pca.ipynb`, but for a series of countries rather than just one.
The quantity of interest is the fraction of policy-time variation captured by the first principal component: high values mean the nine custom OxCGRT fields mostly moved together, while lower values mean more independent policy movement.

Each country is analysed over its OxCGRT simulation window (`find_run_start_time` to `find_run_end_time`). Days with any missing custom field are dropped. PCA is on the max-normalised series in $[0, 1]$, not z-scored.

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import pycountry_convert as pc
from sklearn.decomposition import PCA

from emu_renewal.constants import CONT_CMAP, DATA_PATH, OXCGRT_COLMAP
from emu_renewal.inputs import (
    find_oxcgrt_country_data,
    get_country_pop,
    get_oxcgrt_data,
    get_rel_oxcgrt_cols,
    scale_oxcgrt_pols,
)
from emu_renewal.run import find_run_end_time, find_run_start_time
from emu_renewal.utils import get_cont_of_country, get_country_name

plt.style.use("ggplot")

In [ ]:
countries = json.load(open(DATA_PATH / "config/oxcgrt_included.json"))
oxcgrt = get_oxcgrt_data()

In [ ]:
def pc1_for_country(iso3: str) -> dict:
    start = find_run_start_time(get_country_pop(iso3), iso3)
    end = find_run_end_time(iso3, "oxcgrt")
    pol = find_oxcgrt_country_data(iso3, oxcgrt)
    scaled = scale_oxcgrt_pols(pol[get_rel_oxcgrt_cols("M", pol)])
    policies = scaled[OXCGRT_COLMAP["custom"]]
    time_mask = (policies.index >= pd.Timestamp(start)) & (policies.index <= pd.Timestamp(end))
    policies = policies.loc[time_mask]
    pca_matrix = policies.dropna()
    pca = PCA().fit(pca_matrix)
    return {
        "PC1": pca.explained_variance_ratio_[0],
        "days": pca_matrix.shape[0],
        "start": pd.Timestamp(start).date(),
        "end": pd.Timestamp(end).date(),
    }

rows = {iso3: pc1_for_country(iso3) for iso3 in countries}
pc1 = pd.DataFrame(rows).T
pc1["country"] = pc1.index.map(get_country_name)
pc1["continent"] = pc1.index.map(get_cont_of_country)
pc1 = pc1[["country", "continent", "start", "end", "days", "PC1"]].sort_values("PC1", ascending=False)
pc1["PC1"].describe()

In [ ]:
pc1

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(pc1["PC1"].dropna(), bins=20, range=(0, 1), edgecolor="0.2")
ax.set_xlabel("PC1 share of variance")
ax.set_ylabel("Countries")
ax.set_xlim(0, 1)
ax.set_title("PC1 size, OxCGRT custom fields")
fig.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(9, 5), sharex=True, sharey=True)
for ax, cont in zip(axes.ravel(), CONT_CMAP):
    vals = pc1.loc[pc1["continent"] == cont, "PC1"].dropna()
    ax.hist(vals, bins=12, range=(0, 1), color=CONT_CMAP[cont], edgecolor="0.2")
    ax.set_xlim(0, 1)
    ax.set_title(pc.convert_continent_code_to_continent_name(cont))
fig.suptitle("PC1 share of variance histograms by continent")
plt.show()